## Fisher scoring

In [1]:
# 1. Импорт библиотек
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from fisher_scoring import FisherScoringFocalRegression

# 2. Загрузка очищенных данных
# (Путь '../data/...' корректен, так как блокнот лежит в папке notebooks)
df = pd.read_csv('../data/processed/cleaned_data.csv')

# 3. Разделение на признаки (X) и целевую переменную (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

# 4. Кодирование категориальных признаков (One-Hot Encoding)
# Используем drop_first=True, как в нашей базовой модели, для честного сравнения
X_encoded = pd.get_dummies(X, drop_first=True)

# 5. Разделение на Train/Test (стратифицированно)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Данные готовы. Размер Train: {X_train.shape}, Размер Test: {X_test.shape}")

# 6. Инициализация и обучение модели Fisher Scoring
print("Начинаем обучение Fisher Scoring...")
# gamma=2 — параметр фокусировки на сложных примерах
model_fisher = FisherScoringFocalRegression(gamma=2) 
model_fisher.fit(X_train, y_train)

# 7. Оценка качества модели
try:
    # Пытаемся получить вероятности (predict_proba)
    y_pred_proba_fisher = model_fisher.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, y_pred_proba_fisher)
    print(f"✅ ROC-AUC (Fisher Scoring): {auc_score:.4f}")
except AttributeError:
    # Если в библиотеке нет predict_proba, используем predict (но ROC-AUC может не посчитаться)
    print("⚠️ У модели нет метода predict_proba, считаем метрики по классам...")
    y_pred_fisher = model_fisher.predict(X_test)
    print("Отчет по классификации:")
    from sklearn.metrics import classification_report
    print(classification_report(y_test, y_pred_fisher))

ModuleNotFoundError: No module named 'fisher_scoring'